# PySpark reference 3: types, casting and nulls

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-03-types-casting-and-nulls.ipynb)

Spark data types, decimal arithmetic, `timestamp` vs `timestamp_ntz`, `cast` and `try_cast` under ANSI mode, formatted numbers, null semantics, `fillna`/`dropna`/`replace`, NaN, and typing raw data with reject counts. Practise it in [PySpark practice 3: types, casting and nulls](pyspark-practice-03-types-casting-and-nulls.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Data types](#data-types) | [Spark SQL data types](#spark-sql-data-types), [Decimal precision, scale and arithmetic](#decimal-precision-scale-and-arithmetic), [`timestamp` vs `timestamp_ntz`](#timestamp-vs-timestamp_ntz) |
| [Casting](#casting) | [`Column.cast(dataType)`](#columncastdatatype), [`Column.try_cast(dataType)`](#columntry_castdatatype), [`to_number`, `try_to_number` and format strings](#to_number-try_to_number-and-format-strings), [ANSI mode](#ansi-mode) |
| [Null semantics](#null-semantics) | [Three-valued logic](#three-valued-logic), [`isNull`, `isNotNull` and `eqNullSafe`](#isnull-isnotnull-and-eqnullsafe), [How nulls behave across operations](#how-nulls-behave-across-operations) |
| [DataFrame null methods](#dataframe-null-methods) | [`fillna(value, subset=None)` / `df.na.fill`](#fillnavalue-subsetnone--dfnafill), [`dropna(how="any", thresh=None, subset=None)` / `df.na.drop`](#dropnahowany-threshnone-subsetnone--dfnadrop), [`replace(to_replace, value, subset=None)` / `df.na.replace`](#replaceto_replace-value-subsetnone--dfnareplace) |
| [NaN](#nan) | [NaN semantics](#nan-semantics) |
| [Type checks and data quality](#type-checks-and-data-quality) | [Typing raw data with a reject count](#typing-raw-data-with-a-reject-count) |

## Data types

### Spark SQL data types

Every column has one type from `pyspark.sql.types`. The DDL name is what `cast`, schema strings and SQL use.

| DDL name | Python class | holds | Python value |
| --- | --- | --- | --- |
| `tinyint`, `smallint`, `int`, `bigint` | `ByteType`, `ShortType`, `IntegerType`, `LongType` | 8-, 16-, 32-, 64-bit integers | `int` |
| `float`, `double` | `FloatType`, `DoubleType` | binary floating point | `float` |
| `decimal(p,s)` | `DecimalType(p, s)` | exact numbers, `p` digits with `s` after the point (p ≤ 38) | `decimal.Decimal` |
| `string` | `StringType` | text (optionally with a collation in Spark 4) | `str` |
| `binary` | `BinaryType` | bytes | `bytearray` |
| `boolean` | `BooleanType` | true / false | `bool` |
| `date` | `DateType` | a calendar day | `datetime.date` |
| `timestamp` | `TimestampType` | an instant (UTC internally), shown in the session time zone | `datetime.datetime` |
| `timestamp_ntz` | `TimestampNTZType` | a wall-clock date-time with no time zone | `datetime.datetime` |
| `interval day to second`, `interval year to month` | `DayTimeIntervalType`, `YearMonthIntervalType` | durations | `datetime.timedelta` |
| `array<T>`, `map<K,V>`, `struct<...>` | `ArrayType`, `MapType`, `StructType` | nested values | `list`, `dict`, `Row` |
| `variant` | `VariantType` | semi-structured JSON-like values (Spark 4) | `VariantVal` |

**Notes:** use `decimal` for money and quantities that must add up exactly, `double` for measurements, `bigint` for
ids that may exceed 2.1 billion, and `date` rather than `timestamp` for business dates.

In [ ]:
typed = spark.createDataFrame(
    [(1, 2**40, 0.1, D("0.10"), "x", True, date(2026, 1, 5), datetime(2026, 1, 5, 9, 15))],
    "i int, l bigint, d double, m decimal(12,2), s string, b boolean, dt date, ts timestamp")
typed.select(*[F.expr(f"typeof({c})").alias(c) for c in typed.columns]).show()

### Decimal precision, scale and arithmetic

`decimal(p, s)` stores exactly `p` significant digits, `s` of them after the decimal point, so `decimal(12,2)` holds
values up to 9,999,999,999.99. Arithmetic widens the result type so it cannot overflow: adding two `decimal(12,2)`
gives `decimal(13,2)`; multiplying gives `decimal(25,4)`; `sum` adds 10 digits of precision. Precision is capped at
38; beyond that Spark reduces the scale (`spark.sql.decimalOperations.allowPrecisionLoss`, default true).

**Notes**
- `double` arithmetic is inexact: `0.1 + 0.2` is `0.30000000000000004`, and sums of many doubles drift. Keep money in
  `decimal` end to end; a single `double` in an expression turns the result into `double`. In the Python API
  `F.col("amount") * 1.2` multiplies by a double; in SQL `amount * 1.2` uses a decimal literal and `1.2D` a double.
- A value that does not fit the target precision raises `NUMERIC_VALUE_OUT_OF_RANGE` under ANSI mode.

In [ ]:
nums = spark.createDataFrame([(0.1, 0.2, D("0.10"), D("0.20"))], "a double, b double, c decimal(12,2), d decimal(12,2)")
sums = nums.select((F.col("a") + F.col("b")).alias("double_sum"), (F.col("c") + F.col("d")).alias("decimal_sum"),
                   (F.col("c") * F.col("d")).alias("decimal_product"), (F.col("c") * 1.2).alias("decimal_times_double"),
                   (F.col("c") * F.lit(D("1.2"))).alias("decimal_times_decimal"))
sums.show()
print(sums.dtypes)

### `timestamp` vs `timestamp_ntz`

A `timestamp` is an **instant**: stored as UTC, displayed and converted to dates in the session time zone
(`spark.sql.session.timeZone`), so the same value shows different wall-clock times in different zones. A
`timestamp_ntz` is a **wall-clock reading** with no zone: it shows the same digits everywhere and never shifts.

**Use them for:** `timestamp` for events (orders, logs); `timestamp_ntz` for local schedules (store opening hours,
"9:00 local") and for sources that send times without an offset you do not want reinterpreted.

**Notes:** set the session time zone explicitly in every job; the cluster default otherwise decides every derived
date. Time-zone conversions are covered in [PySpark reference 3: types, casting and nulls](pyspark-reference-03-types-casting-and-nulls.ipynb).

In [ ]:
both = spark.createDataFrame([(datetime(2026, 1, 14, 0, 30), datetime(2026, 1, 14, 0, 30))], "ts timestamp, ntz timestamp_ntz")
for zone in ["UTC", "America/Chicago"]:
    spark.conf.set("spark.sql.session.timeZone", zone)
    both.select(F.lit(zone).alias("session_zone"), F.col("ts").cast("string").alias("timestamp"),
                F.col("ntz").cast("string").alias("timestamp_ntz"), F.to_date("ts").alias("date_of_ts")).show()
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Casting

### `Column.cast(dataType)`

Converts a column to another type. `dataType` is a DDL string (`"int"`, `"decimal(12,2)"`, `"date"`) or a `DataType`
object. Under Spark 4's default **ANSI mode**, a value that cannot be converted raises an error and fails the job;
with ANSI off it becomes null.

| from → to | behaviour |
| --- | --- |
| string → int / decimal / double | parses the text; leading and trailing spaces are allowed; `"1,204.50"`, `"N/A"` and `""` fail |
| string → date / timestamp | ISO formats only (`2026-01-31`, `2026-01-31 18:45:00`); other formats need `to_date(col, fmt)` |
| double → int | truncates toward zero; out-of-range values fail |
| int → string, date → string | always succeeds |
| string → boolean | `true/false`, `t/f`, `yes/no`, `y/n`, `1/0` (case-insensitive) |
| numeric → decimal(p,s) | rounds to scale `s` (half up); too many integer digits fail |

**Returns:** a Column of the target type.

**Notes:** casting is the usual place a pipeline breaks on one bad value. Use `try_cast` where bad values are
expected, then count the nulls it produced.

In [ ]:
raw = spark.createDataFrame([(" 42 ", "2026-01-31", "3.9", "yes"), ("7", "2026-01-31 18:45:00", "-3.9", "0")], "n string, d string, x string, flag string")
raw.select(F.col("n").cast("int").alias("int"), F.col("d").cast("date").alias("date"), F.col("x").cast("double").cast("int").alias("truncated"),
           F.col("x").cast("decimal(4,0)").alias("rounded"), F.col("flag").cast("boolean").alias("bool")).show()
with expected_error():
    spark.createDataFrame([("N/A",)], "n string").select(F.col("n").cast("int")).collect()

### `Column.try_cast(dataType)`

The same conversion as `cast`, but a value that cannot be converted becomes **null** instead of failing the job
(`Column.try_cast` in PySpark 4.0; `F.expr("try_cast(c AS INT)")` or SQL `try_cast` in older versions).

**Use it for:** typing raw strings in a bronze-to-silver step where some values are known to be bad, combined with a
reject count or quarantine.

**Notes:** a null from `try_cast` can mean "missing in the source" or "present but invalid". Keep the raw column until
you have separated the two: `raw IS NOT NULL AND typed IS NULL` is a reject.

In [ ]:
amounts = spark.createDataFrame([("o1", "120.50"), ("o2", "N/A"), ("o3", None), ("o4", "1,204.50")], "id string, amount_text string")
typed_amounts = amounts.withColumn("amount", F.col("amount_text").try_cast("decimal(12,2)"))
typed_amounts.withColumn("rejected", F.col("amount_text").isNotNull() & F.col("amount").isNull()).show()

### `to_number`, `try_to_number` and format strings

Parse text that follows a number **format**: thousands separators, currency signs, signs in brackets.
`F.to_number(col, format)` fails on a mismatch; `F.try_to_number(col, format)` returns null.

| format element | meaning |
| --- | --- |
| `9` / `0` | a digit (`0` means it must be present) |
| `.` / `D` | decimal point |
| `,` / `G` | grouping separator |
| `$` | currency sign |
| `S` / `MI` | sign at the start or end |
| `PR` | negative value in angle brackets |

**Returns:** a decimal column.

In [ ]:
formatted = spark.createDataFrame([("1,204.50",), ("$99.00",), ("12",), ("N/A",)], "text string")
formatted.select("text", F.try_to_number("text", F.lit("9,999.99")).alias("plain_format"),
                 F.try_to_number("text", F.lit("$999.99")).alias("dollar_format")).show()

### ANSI mode

`spark.sql.ansi.enabled` (default **true** in Spark 4, false in Spark 3) makes Spark follow the SQL standard for
invalid operations: it raises an error instead of quietly returning null or a wrapped value.

| operation | ANSI on (Spark 4) | ANSI off |
| --- | --- | --- |
| invalid cast (`'N/A'` to int) | error `CAST_INVALID_INPUT` | null |
| integer overflow (`2147483647 + 1`) | error `ARITHMETIC_OVERFLOW` | wraps around to a negative number |
| division by zero | error `DIVIDE_BY_ZERO` | null |
| array index out of range in `element_at` | error | null |
| implicit string-to-number comparison with a bad value | error | null, row does not match |

The `try_` functions (`try_cast`, `try_add`, `try_subtract`, `try_multiply`, `try_divide`, `try_sum`, `try_avg`,
`try_element_at`, `try_to_number`, `try_to_timestamp`) give the lenient behaviour per expression while ANSI stays on.

**Notes:** upgrading a pipeline from Spark 3 to 4 most often breaks here; turning ANSI off globally hides data
problems, so prefer fixing the expressions.

In [ ]:
limits = spark.createDataFrame([(2147483647, 0)], "big int, zero int")
with expected_error():
    limits.select(F.col("big") + 1).collect()
with expected_error():
    limits.select(F.col("big") / F.col("zero")).collect()
limits.select(F.try_add("big", F.lit(1)).alias("try_add"), F.try_divide("big", "zero").alias("try_divide")).show()
spark.conf.set("spark.sql.ansi.enabled", "false")
limits.select((F.col("big") + 1).alias("ansi_off_overflow"), (F.col("big") / F.col("zero")).alias("ansi_off_divide")).show()
spark.conf.set("spark.sql.ansi.enabled", "true")

## Null semantics

### Three-valued logic

SQL null means "unknown". Any comparison or arithmetic with null returns null, and boolean logic has three values:

| a | b | `a AND b` | `a OR b` | `NOT a` |
| --- | --- | --- | --- | --- |
| true | null | null | **true** | false |
| false | null | **false** | null | true |
| null | null | null | null | null |

A filter keeps rows where the condition is **true**, so rows where it is null disappear from both `where(cond)` and
`where(~cond)`.

**Notes:** `col == None` and `col != None` are always null and select nothing; use `isNull()` / `isNotNull()`.

In [ ]:
total = customers.count()
north = customers.where(F.col("region") == "North").count()
not_north = customers.where(~(F.col("region") == "North")).count()
print(f"rows: {total} | region = North: {north} | NOT region = North: {not_north} | lost to null: {total - north - not_north}")
print("region == None selects:", customers.where(F.col("region") == None).count(), "| isNull selects:", customers.where(F.col("region").isNull()).count())  # noqa: E711

### `isNull`, `isNotNull` and `eqNullSafe`

| expression | null input | use for |
| --- | --- | --- |
| `col.isNull()` / `F.isnull(col)` | true | finding missing values |
| `col.isNotNull()` / `F.isnotnull(col)` | false | required-field checks |
| `a.eqNullSafe(b)` (SQL `a <=> b`) | null `<=>` null is true; null `<=>` value is false | comparing two nullable columns: change detection, joins on nullable keys |

**Notes:** `~a.eqNullSafe(b)` is the null-safe "is different" test, the right condition for "did this attribute
change between two snapshots".

In [ ]:
pairs = spark.createDataFrame([("a", "a"), ("a", "b"), ("a", None), (None, None)], "old string, new string")
pairs.select("old", "new", (F.col("old") == F.col("new")).alias("equals"), F.col("old").eqNullSafe(F.col("new")).alias("null_safe_equals"),
             (~F.col("old").eqNullSafe(F.col("new"))).alias("changed")).show()

### How nulls behave across operations

| operation | null behaviour |
| --- | --- |
| arithmetic, string functions | null in → null out (`concat_ws` and `coalesce` are exceptions) |
| `count(*)` | counts rows, including all-null rows |
| `count(col)`, `sum`, `avg`, `min`, `max` | skip nulls; over only nulls they return null (count returns 0) |
| `groupBy(col)` | all null keys form **one** group |
| `distinct`, `dropDuplicates` | nulls are equal to each other |
| `orderBy` | nulls first ascending, last descending |
| join on `==` | null keys never match |
| `union` | nulls kept as is |
| `when(cond, ...)` | a null condition counts as false |

In [ ]:
customers.groupBy("region").agg(F.count("*").alias("rows")).orderBy(F.asc_nulls_last("region")).show()
spark.createDataFrame([(None,), (None,)], "v int").agg(F.count("*").alias("count_star"), F.count("v").alias("count_v"), F.sum("v").alias("sum_v")).show()

## DataFrame null methods

### `fillna(value, subset=None)` / `df.na.fill`

Replaces nulls (and `NaN` in float and double columns) with a value.

| parameter | type | description |
| --- | --- | --- |
| `value` | int, float, str, bool, or dict | the replacement; a dict maps column names to values |
| `subset` | str or list of str | columns to fill (ignored when `value` is a dict) |

**Returns:** a DataFrame.

**Notes:** a scalar `value` is applied only to columns of a **matching type**: `fillna(0)` fills numeric columns and
silently skips string columns; `fillna("unknown")` fills only string columns. Use a dict to be explicit. Filling a
numeric null with 0 changes averages; do it only when missing really means zero.

In [ ]:
gaps = spark.createDataFrame([(1, None, None, None), (2, "web", 5, 1.5)], "id int, channel string, qty int, score double")
gaps.fillna(0).show()
gaps.fillna("unknown").show()
gaps.fillna({"channel": "unknown", "qty": 0}).show()

### `dropna(how="any", thresh=None, subset=None)` / `df.na.drop`

Removes rows with nulls (and `NaN`).

| parameter | description |
| --- | --- |
| `how` | `"any"`: drop if any of the considered columns is null; `"all"`: drop only if all are null |
| `thresh` | keep rows with at least this many non-null values; overrides `how` |
| `subset` | columns to consider (default: all) |

**Notes:** `dropna()` with no arguments drops a row for a null in any column, often far more than intended. Name the
required columns.

In [ ]:
contacts = spark.createDataFrame([(1, "555-01", None), (2, None, None), (3, None, "a@x.com")], "id int, phone string, email string")
print("any:", [r.id for r in contacts.dropna().collect()],
      "| all phone/email:", [r.id for r in contacts.dropna(how="all", subset=["phone", "email"]).collect()],
      "| thresh=2:", [r.id for r in contacts.dropna(thresh=2).collect()])

### `replace(to_replace, value, subset=None)` / `df.na.replace`

Replaces exact values (not nulls by default) across columns. `to_replace` can be a value, a list (with a list of
`value`s of the same length) or a dict `{old: new}`. Replacing with `None` turns sentinels into nulls.

**Use it for:** source sentinels such as `-1`, `""`, `"N/A"`, `"null"`.

**Notes:** like `fillna`, it only touches columns whose type matches the values; mixing types in one call is not
allowed.

In [ ]:
sentinels = spark.createDataFrame([(1, "N/A", -1), (2, "web", 3), (3, "", -1)], "id int, channel string, qty int")
sentinels.replace(["N/A", ""], None, subset=["channel"]).replace(-1, None, subset=["qty"]).show()

## NaN

### NaN semantics

`NaN` (not a number) exists only in `float` and `double` columns, typically from `0.0 / 0.0` in a source system.
Spark treats it differently from both null and the IEEE standard:

| rule | Spark behaviour |
| --- | --- |
| `NaN = NaN` | **true** (unlike IEEE) |
| ordering | `NaN` is larger than any other number, so it sorts last ascending |
| `isnull(NaN)` | false; only `isnan` finds it |
| aggregates | one `NaN` makes `sum` and `avg` return `NaN` |
| `fillna`, `dropna` | treat `NaN` like null in float and double columns |
| `nanvl(col, x)` | replaces `NaN` with `x` |

In [ ]:
readings = spark.createDataFrame([(1.5,), (float("nan"),), (None,), (3.0,)], "v double")
readings.select("v", F.isnan("v").alias("isnan"), F.isnull("v").alias("isnull"), (F.col("v") == float("nan")).alias("equals_nan")).show()
readings.agg(F.avg("v").alias("avg"), F.avg(F.nanvl("v", F.lit(None).cast("double"))).alias("avg_without_nan"), F.max("v").alias("max")).show()
print("sorted:", [r.v for r in readings.orderBy("v").collect()])

## Type checks and data quality

### Typing raw data with a reject count

A standard bronze-to-silver step: cast every raw string with `try_cast` (or `try_to_timestamp`, `try_to_number`),
keep the raw value, and count rows where a present value failed to convert. Route rejects to a quarantine table and
fail the job when the reject rate crosses a threshold.

```python
typed = raw.select(*keys, *[F.col(c).try_cast(t).alias(c) for c, t in targets.items()],
                   *[F.col(c).alias(f"_raw_{c}") for c in targets])
reject = F.lit(False)
for c in targets:
    reject = reject | (F.col(f"_raw_{c}").isNotNull() & F.col(c).isNull())
```

**Notes:** `typeof(col)` (SQL function) returns the runtime type name of an expression, useful when debugging
implicit casts.

In [ ]:
landing = spark.createDataFrame([("o1", "4", "2026-01-05"), ("o2", "x", "2026-01-06"), ("o3", "2", "05/01/2026"), ("o4", None, None)],
                                "order_id string, qty string, order_date string")
targets = {"qty": "int", "order_date": "date"}
typed = landing.select("order_id", *[F.col(c).try_cast(t).alias(c) for c, t in targets.items()], *[F.col(c).alias(f"_raw_{c}") for c in targets])
reject = F.lit(False)
for c in targets:
    reject = reject | (F.col(f"_raw_{c}").isNotNull() & F.col(c).isNull())
checked = typed.withColumn("is_reject", reject)
checked.show()
print("reject rate:", checked.where("is_reject").count() / checked.count())

**Next:** practise these entries in [PySpark practice 3: types, casting and nulls](pyspark-practice-03-types-casting-and-nulls.ipynb).